<a href="https://colab.research.google.com/github/marsanla/C4L/blob/main/SOL_C4L_5_Ejercicios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Soluciones — Módulo 5: Ejercicios consolidados

> Soluciones de los 10 ejercicios de C4L_5. Todas usan **funciones**, manejo seguro de archivos (`with open`), validación de entradas y **comentarios línea a línea** para que se entiendan paso a paso.


## Preparación — generar los archivos de muestra


In [1]:
# Estos son los mismos archivos que en el notebook de ejercicios.
import pandas as pd

datos_contratos = {
    "ContratoID":      [101, 102, 103, 104, 105],
    "TipoContrato":    ["Arrendamiento", "Servicio", "Compra", "Consultoría", "Licencia"],
    "FechaInicio":     ["2021-01-01", "2021-03-15", "2021-05-20", "2021-07-01", "2021-09-05"],
    "FechaVencimiento":["2024-01-01", "2023-03-15", "2023-05-20", "2024-07-01", "2026-09-05"],
    "MontoContrato":   [1500, 3000, 25000, 8500, 12000],
    "Estado":          ["Activo", "Activo", "Pendiente", "Activo", "Expirado"]
}
pd.DataFrame(datos_contratos).to_csv("contratos.csv", index=False)


texto_legislacion_spain = '''
Artículo 1. El propósito de esta ley es asegurar los derechos de los trabajadores.
Artículo 2. Todos los empleadores deben seguir las normas de seguridad establecidas en el código laboral.
Artículo 3. La jornada laboral máxima sin horas extra no debe exceder de 40 horas semanales.
'''
texto_legislacion_france = '''
Artículo 1. El propósito de esta ley es asegurar los derechos de los trabajadores.
Artículo 2. Todos los empleadores deben seguir las normas de seguridad establecidas en el código laboral.
Artículo 3. La durée maximale du travail sans heures supplémentaires ne doit pas dépasser 35 heures par semaine.
'''
with open("legislacion_1.txt", "w", encoding="utf-8") as f:
    f.write(texto_legislacion_spain)
with open("legislacion_2.txt", "w", encoding="utf-8") as f:
    f.write(texto_legislacion_france)


casos = {
    "CasoID":          [1, 2, 3, 4, 5],
    "TipoCaso":        ["Civil", "Penal", "Comercial", "Familia", "Laboral"],
    "TotalDisputa":    [50000, 150000, 75000, 30000, 60000],
    "Estado":          ["Ganado", "Perdido", "Pendiente", "Ganado", "Perdido"],
    "FechaProceso":    ["2023-04-15", "2023-04-18", "2023-04-20", "2023-03-22", "2023-04-25"],
    "FechaInicio":     ["2021-01-10", "2021-02-20", "2021-03-15", "2021-04-22", "2021-05-30"],
    "FechaResolucion": ["2022-01-15", "2022-02-25", "Pendiente", "2022-05-05", "2022-06-10"]
}
pd.DataFrame(casos).to_csv("casos.csv", index=False)
print("Archivos de muestra creados: contratos.csv, casos.csv, legislacion_1.txt, legislacion_2.txt")


Archivos de muestra creados: contratos.csv, casos.csv, legislacion_1.txt, legislacion_2.txt


## Soluciones


### Ejercicio 1 — Calculadora de indemnizaciones


In [2]:
def calcular_indemnizacion(años_trabajados, salario_mensual):
    """Indemnización simplificada: 1 mes de salario por año trabajado."""
    return años_trabajados * salario_mensual


# Pruebas
print(calcular_indemnizacion(5, 3000))    # 15000
print(calcular_indemnizacion(10, 2500))   # 25000

# Explicación: una multiplicación. La función encapsula la fórmula y le da nombre.


15000
25000


### Ejercicio 2 — Verificador de vencimientos


In [3]:
import pandas as pd
from datetime import datetime, timedelta

def verificar_fechas_vencimiento(archivo_csv):
    """Devuelve los contratos que vencen en los próximos 30 días."""
    df = pd.read_csv(archivo_csv)

    # Convertir a datetime
    df["FechaVencimiento"] = pd.to_datetime(df["FechaVencimiento"])

    # Definir el rango: desde hoy hasta dentro de 30 días
    hoy = datetime.now()
    limite = hoy + timedelta(days=30)

    # Filtrar
    proximos = df[(df["FechaVencimiento"] >= hoy) & (df["FechaVencimiento"] <= limite)]
    return proximos


print(verificar_fechas_vencimiento("contratos.csv"))

# Explicación:
# - pd.to_datetime convierte la columna a fechas.
# - El filtro combina dos condiciones con `&` (Y lógico en pandas).
# - Recuerda los paréntesis alrededor de cada condición.


Empty DataFrame
Columns: [ContratoID, TipoContrato, FechaInicio, FechaVencimiento, MontoContrato, Estado]
Index: []


### Ejercicio 3 — Clasificador de documentos


In [4]:
def clasificar_documentos(texto):
    """Clasifica un texto legal por palabras clave."""
    texto_lower = texto.lower()
    if "contrato" in texto_lower:
        return "Contratos"
    elif "demanda" in texto_lower:
        return "Demandas"
    elif "sentencia" in texto_lower:
        return "Sentencias"
    else:
        return "Otros"


# Pruebas
print(clasificar_documentos("Este documento es un contrato de servicio."))
print(clasificar_documentos("La demanda fue presentada el 15 de marzo."))
print(clasificar_documentos("La sentencia es firme."))
print(clasificar_documentos("Comunicación interna del despacho."))

# Explicación:
# - Pasamos a minúsculas UNA SOLA VEZ y reutilizamos la variable.
# - El `in` comprueba si la palabra aparece como subcadena.


Contratos
Demandas
Sentencias
Otros


### Ejercicio 4 — Registro de horas en JSON


In [5]:
import json

def registrar_horas(archivo="registro_horas.json"):
    """Pide datos al usuario y los guarda en un JSON."""
    abogado_id = input("Introduce el ID del abogado: ")
    caso_id    = input("Introduce el ID del caso: ")
    horas_str  = input("Horas trabajadas: ")

    # Validación básica del campo numérico
    try:
        horas = float(horas_str)
    except ValueError:
        print("ERROR: las horas deben ser un número. Cancelando.")
        return None

    registro = {
        "abogado_id": abogado_id,
        "caso_id":    caso_id,
        "horas":      horas,
    }

    with open(archivo, "w", encoding="utf-8") as f:
        json.dump(registro, f, ensure_ascii=False, indent=2)

    print(f"Registro guardado en {archivo}")
    return registro


# Para probar sin pedir input al usuario, sustituimos input() por valores fijos:
def registrar_horas_demo():
    registro = {"abogado_id": "ABG-001", "caso_id": "CASO-2024-15", "horas": 4.5}
    with open("registro_horas.json", "w", encoding="utf-8") as f:
        json.dump(registro, f, ensure_ascii=False, indent=2)
    return registro


registrar_horas_demo()

# Verificación
with open("registro_horas.json", "r", encoding="utf-8") as f:
    print(json.load(f))


{'abogado_id': 'ABG-001', 'caso_id': 'CASO-2024-15', 'horas': 4.5}


### Ejercicio 5 — Calculadora de tasas judiciales


In [6]:
def calcular_tasas_judiciales(total_disputa, tipo_procedimiento):
    """Devuelve la tasa judicial según el tipo de procedimiento."""
    porcentajes = {
        "civil":     0.020,
        "comercial": 0.030,
        "penal":     0.015,
        "familia":   0.010,
    }
    porcentaje = porcentajes.get(tipo_procedimiento.lower())
    if porcentaje is None:
        return "Tipo de procedimiento no reconocido"
    return total_disputa * porcentaje


# Pruebas
print(calcular_tasas_judiciales(10000, "civil"))      # 200.0
print(calcular_tasas_judiciales(20000, "comercial"))  # 600.0
print(calcular_tasas_judiciales(5000,  "penal"))      # 75.0
print(calcular_tasas_judiciales(15000, "familia"))    # 150.0
print(calcular_tasas_judiciales(1000,  "constitucional"))  # error

# Explicación:
# - Diccionario tipo -> porcentaje. .get() devuelve None si la clave no existe.
# - Validamos el resultado y devolvemos un mensaje en lugar de fallar.


200.0
600.0
75.0
150.0
Tipo de procedimiento no reconocido


### Ejercicio 6 — Analizador de riesgo


In [7]:
def evaluar_riesgo_legal(naturaleza_caso, jurisdiccion):
    """Devuelve 'Bajo', 'Medio' o 'Alto' según la combinación."""
    naturaleza = naturaleza_caso.lower()
    jurisdic   = jurisdiccion.lower()

    # Reglas explícitas (cualquier combinación no listada -> Medio)
    reglas = {
        ("comercial", "nacional"): "Alto",
        ("penal",     "estatal"):  "Alto",
        ("penal",     "nacional"): "Alto",
        ("familia",   "nacional"): "Medio",
        ("civil",     "local"):    "Bajo",
    }
    return reglas.get((naturaleza, jurisdic), "Medio")


print(evaluar_riesgo_legal("civil", "local"))         # Bajo
print(evaluar_riesgo_legal("comercial", "nacional"))  # Alto
print(evaluar_riesgo_legal("penal", "estatal"))       # Alto
print(evaluar_riesgo_legal("familia", "nacional"))    # Medio
print(evaluar_riesgo_legal("laboral", "local"))       # Medio (default)

# Explicación:
# - Una tabla de reglas con TUPLAS como claves es muy elegante para
#   reglas con dos dimensiones.
# - .get((clave_a, clave_b), default) devuelve 'Medio' si la combinación
#   no está en la tabla.


Bajo
Alto
Alto
Medio
Medio


### Ejercicio 7 — Informe mensual de casos


In [8]:
import pandas as pd
from datetime import datetime

def generar_informe_casos(datos_casos, mes, anio, archivo_salida):
    """Filtra los casos del mes/año indicados y guarda el informe en CSV."""
    # Asegurarse de que la columna sea datetime
    datos_casos = datos_casos.copy()
    datos_casos["FechaProceso"] = pd.to_datetime(datos_casos["FechaProceso"])

    # Filtrar por mes y año
    filtrados = datos_casos[
        (datos_casos["FechaProceso"].dt.month == mes) &
        (datos_casos["FechaProceso"].dt.year  == anio)
    ]

    filtrados.to_csv(archivo_salida, index=False)
    print(f"Informe guardado en {archivo_salida} con {len(filtrados)} casos.")
    return filtrados


datos = {
    "TipoCaso":     ["Civil", "Comercial", "Penal", "Civil", "Comercial"],
    "Estado":       ["Resuelto", "Pendiente", "Resuelto", "Resuelto", "Pendiente"],
    "TotalDisputa": [5000, 20000, 15000, 7500, 10000],
    "FechaProceso": ["2023-04-15", "2023-04-18", "2023-04-20", "2023-03-22", "2023-04-25"]
}
casos_df = pd.DataFrame(datos)

informe = generar_informe_casos(casos_df, 4, 2023, "informe_casos_abril_2023.csv")
print(informe)

# Explicación:
# - .dt.month y .dt.year extraen el mes y año de cada fecha.
# - .copy() para no modificar el DataFrame original.


Informe guardado en informe_casos_abril_2023.csv con 4 casos.
    TipoCaso     Estado  TotalDisputa FechaProceso
0      Civil   Resuelto          5000   2023-04-15
1  Comercial  Pendiente         20000   2023-04-18
2      Penal   Resuelto         15000   2023-04-20
4  Comercial  Pendiente         10000   2023-04-25


### Ejercicio 8 — Comparador de legislaciones


In [9]:
from difflib import unified_diff

def comparar_legislaciones(archivo_1, archivo_2):
    """Imprime las diferencias entre dos archivos de texto."""
    with open(archivo_1, "r", encoding="utf-8") as f:
        lineas_1 = f.readlines()
    with open(archivo_2, "r", encoding="utf-8") as f:
        lineas_2 = f.readlines()

    diff = unified_diff(
        lineas_1,
        lineas_2,
        fromfile=archivo_1,
        tofile=archivo_2,
        lineterm=""
    )

    diferencias_encontradas = False
    for linea in diff:
        diferencias_encontradas = True
        print(linea)

    if not diferencias_encontradas:
        print("Los dos archivos son idénticos.")


comparar_legislaciones("legislacion_1.txt", "legislacion_2.txt")

# Explicación:
# - .readlines() devuelve una lista con cada línea del archivo.
# - unified_diff produce un diff estilo git: líneas con `-` están en el primero,
#   con `+` en el segundo.


--- legislacion_1.txt
+++ legislacion_2.txt
@@ -1,4 +1,4 @@
 

 Artículo 1. El propósito de esta ley es asegurar los derechos de los trabajadores.

 Artículo 2. Todos los empleadores deben seguir las normas de seguridad establecidas en el código laboral.

-Artículo 3. La jornada laboral máxima sin horas extra no debe exceder de 40 horas semanales.

+Artículo 3. La durée maximale du travail sans heures supplémentaires ne doit pas dépasser 35 heures par semaine.



### Ejercicio 9 — Estadísticas de casos


In [10]:
import pandas as pd

def contar_casos_ganados_perdidos(archivo_csv):
    """Devuelve el conteo de casos por estado."""
    df = pd.read_csv(archivo_csv)
    return df["Estado"].value_counts().to_dict()


print(contar_casos_ganados_perdidos("casos.csv"))

# Explicación:
# - .value_counts() devuelve una pandas.Series con la frecuencia.
# - .to_dict() la convierte en diccionario para uso fácil.


{'Ganado': 2, 'Perdido': 2, 'Pendiente': 1}


### Ejercicio 10 — Simulador de resultado


In [11]:
import random

def simular_resultado_caso(experiencia_abogado, complejidad_caso):
    """Devuelve 'Ganado' o 'Perdido' usando una probabilidad calculada."""
    # Probabilidad base 0.5, modificada por (experiencia - complejidad) * 0.05
    probabilidad = 0.5 + (experiencia_abogado - complejidad_caso) * 0.05
    # Limitar al rango [0.1, 0.9]
    probabilidad = max(0.1, min(0.9, probabilidad))

    return "Ganado" if random.random() < probabilidad else "Perdido"


# Pruebas
random.seed(42)   # para que las pruebas sean reproducibles
for _ in range(5):
    resultado = simular_resultado_caso(experiencia_abogado=10, complejidad_caso=5)
    print(f"Resultado: {resultado}")

# Explicación:
# - max(0.1, min(0.9, p)) acota el valor al rango [0.1, 0.9] (técnica clamp).
# - random.random() devuelve un float entre 0.0 y 1.0.
# - Si ese float está por DEBAJO de la probabilidad, gana — modela
#   correctamente la probabilidad indicada.


Resultado: Ganado
Resultado: Ganado
Resultado: Ganado
Resultado: Ganado
Resultado: Ganado


## Cierra del módulo

Si llegaste a esta sección habiendo resuelto los ejercicios por tu cuenta o entendiendo cada solución, ya estás **listo para el proyecto final** del módulo C4L_6 — un sistema completo de alertas y reporting para un despacho.
